In [ ]:
import sys

import os
import re
import click
import tqdm
import pickle
import numpy as np
import torch
import PIL.Image
import dnnlib
from torch_utils import distributed as dist
from PIL import Image
from tqdm import tqdm
from project_utils.paths import PROJECT_ROOT, CONFIG_DIR, DATA_DIR

from matplotlib import pyplot as plt


In [ ]:
def edm_sampler(
	net, latents, conditions, class_labels=None, randn_like=torch.randn_like,
	num_steps=18, sigma_min=0.002, sigma_max=80, rho=7,
	S_churn=0, S_min=0, S_max=float('inf'), S_noise=1,
):
	# Adjust noise levels based on what's supported by the network.
	sigma_min = max(sigma_min, net.sigma_min)
	sigma_max = min(sigma_max, net.sigma_max)

	# Time step discretization.
	step_indices = torch.arange(num_steps, dtype=torch.float64, device=latents.device)
	t_steps = (sigma_max ** (1 / rho) + step_indices / (num_steps - 1) * (sigma_min ** (1 / rho) - sigma_max ** (1 / rho))) ** rho
	t_steps = torch.cat([net.round_sigma(t_steps), torch.zeros_like(t_steps[:1])]) # t_N = 0

	# Main sampling loop.
	x_next = latents.to(torch.float64) * t_steps[0]
	for i, (t_cur, t_next) in enumerate(zip(t_steps[:-1], t_steps[1:])): # 0, ..., N-1
		x_cur = x_next

		# Increase noise temporarily.
		gamma = min(S_churn / num_steps, np.sqrt(2) - 1) if S_min <= t_cur <= S_max else 0
		t_hat = net.round_sigma(t_cur + gamma * t_cur)
		x_hat = x_cur + (t_hat ** 2 - t_cur ** 2).sqrt() * S_noise * randn_like(x_cur)

		# Euler step.
		denoised = net(x_hat, t_hat, conditions, class_labels).to(torch.float64)
		d_cur = (x_hat - denoised) / t_hat
		x_next = x_hat + (t_next - t_hat) * d_cur

		# Apply 2nd order correction.
		if i < num_steps - 1:
			denoised = net(x_next, t_next, conditions, class_labels).to(torch.float64)
			d_prime = (x_next - denoised) / t_next
			x_next = x_hat + (t_next - t_hat) * (0.5 * d_cur + 0.5 * d_prime)

	return x_next



#----------------------------------------------------------------------------
# Wrapper for torch.Generator that allows specifying a different random seed
# for each sample in a minibatch.

class StackedRandomGenerator:
	def __init__(self, device, seeds):
		super().__init__()
		self.generators = [torch.Generator(device).manual_seed(int(seed) % (1 << 32)) for seed in seeds]

	def randn(self, size, **kwargs):
		assert size[0] == len(self.generators)
		return torch.stack([torch.randn(size[1:], generator=gen, **kwargs) for gen in self.generators])

	def randn_like(self, input):
		return self.randn(input.shape, dtype=input.dtype, layout=input.layout, device=input.device)

	def randint(self, *args, size, **kwargs):
		assert size[0] == len(self.generators)
		return torch.stack([torch.randint(*args, size=size[1:], generator=gen, **kwargs) for gen in self.generators])

#----------------------------------------------------------------------------
# Parse a comma separated list of numbers or ranges and return a list of ints.
# Example: '1,2,5-10' returns [1, 2, 5, 6, 7, 8, 9, 10]

def parse_int_list(s):
	if isinstance(s, list): return s
	ranges = []
	range_re = re.compile(r'^(\d+)-(\d+)$')
	for p in s.split(','):
		m = range_re.match(p)
		if m:
			ranges.extend(range(int(m.group(1)), int(m.group(2))+1))
		else:
			ranges.append(int(p))
	return ranges

#----------------------------------------------------------------------------

In [ ]:
# device 
device=torch.device('cuda:0')


network_pkl = '/home/zez0002/cond-diff-pou/outputs/00002-SQG_32_dynamic-medium-gpus1-batch256-fp32/network-snapshot-001024.pkl'
# Load network.
print(f'Loading network from "{network_pkl}"...')
with dnnlib.util.open_url(network_pkl, verbose=True) as f:
	net_32_dy = pickle.load(f)['ema'].to(device)


In [ ]:
# arguemnt variblas for the script
# Variables with default values
outdir = 'output'       # Required, so assign path before running
subdirs = False
class_idx = None
max_batch_size = 50
seeds = parse_int_list('0-49')


# batch size
num_batches = ((len(seeds) - 1) // (max_batch_size ) + 1) 
all_batches = torch.as_tensor(seeds).tensor_split(num_batches)
rank_batches = all_batches
batch_seeds = all_batches[0]
batch_size = len(batch_seeds)

# Loop over batches.
print(f'Generating {len(seeds)} images to "{outdir}"...')


# Pick latents and labels.
rnd = StackedRandomGenerator(device, batch_seeds)


In [ ]:
# load some data for initialization
import zipfile
import torch
import io

dataset_dir = os.path.join(DATA_DIR,"SQG", 'SQG_v_0_lo_32_c_2', 'SQG_v_0_lo_32_c_2.zip') 

n_sample_check = 100
feild_32 = []
with zipfile.ZipFile(dataset_dir, 'r') as zf:
	for i in range(n_sample_check):
		with zf.open(f'sample_{i:06d}.pt') as f:
			data = torch.load(io.BytesIO(f.read()))
			feild_32.append(data['targets'])

feild_32 = torch.concat(feild_32, dim=0)
print(feild_32.shape)

In [ ]:

from IPython.display import HTML
import matplotlib.animation as animation
import matplotlib as mpl
mpl.rcParams['animation.embed_limit'] = 30 # in MB (30 MB in this example)


# your existing setup...
n_frame_show = n_sample_check
fields_all = feild_32
n_start = 0 
nframes = n_frame_show
total_duration = n_frame_show * 0.1

def update(i):
	i = i + n_start
	mat0.set_data(fields_all[i])
	ax.set_title(f'pv_all_0 (idx = {i}) ')

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 5))
mat0 = ax.imshow(fields_all[n_start], vmax=1, vmin=-1)
fig.colorbar(mat0, ax=ax)
fig.tight_layout()



ani = animation.FuncAnimation(
	fig, update, frames=nframes, interval=int(1000.0 * total_duration / nframes)
)

plt.close(fig)

# Display the animation inline
HTML(ani.to_jshtml())  # or use ani.to_html5_video()


In [ ]:
# generat a single path 
batch_seeds = torch.tensor([0], device=device)
rnd = StackedRandomGenerator(device, batch_seeds)
n_pred_step = 100


# condition generation
fields_dm = feild_32[0:2,...].to(device=device) 


In [ ]:
print('initial conditioning:')
print(fields_dm.shape)

In [ ]:
# generation loop
for i in tqdm(range(n_pred_step), desc="Generating 32x32 turb fields"):
	latents = rnd.randn([1, 1, 32, 32], device=device)
	conditions = fields_dm[-2:,...][None,...]
	samples = edm_sampler(net_32_dy, latents, conditions)
	fields_dm = torch.cat((fields_dm, samples[0]), dim=0)

print(fields_dm.shape)

In [ ]:
# visualize 
# your existing setup...
n_frame_show = n_pred_step
fields_all = fields_dm.cpu().numpy()
n_start = 0 
nframes = n_frame_show
total_duration = n_frame_show * 0.1

def update(i):
	i = i + n_start
	mat0.set_data(fields_all[i,...])
	ax.set_title(f'pv_all_0 (idx = {i}) ')

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6, 5))
mat0 = ax.imshow(fields_all[n_start,...], vmax=1, vmin=-1)
fig.colorbar(mat0, ax=ax)
fig.tight_layout()



ani = animation.FuncAnimation(
	fig, update, frames=nframes, interval=int(1000.0 * total_duration / nframes)
)

plt.close(fig)

# Display the animation inline
HTML(ani.to_jshtml())  # or use ani.to_html5_video()


In [ ]:
# generate two path with the same initial condition
# generat a single path 
batch_seeds = torch.tensor([0,1], device=device)
rnd = StackedRandomGenerator(device, batch_seeds)
conditions = feild_32[[0,0],None,...].to(device=device)

print(conditions.shape)


n_pred_step = 100
field_dm = [conditions]
for i in tqdm(range(n_pred_step), desc="Generating 32x32 turb fields"):
	latents = rnd.randn([2, 1, 32, 32], device=device)
	samples = edm_sampler(net_32_dy, latents, conditions)
	conditions = samples.detach().clone()
	field_dm.append(conditions)

field_dm = torch.stack(field_dm, dim=0).cpu().numpy()
print(field_dm.shape)

In [ ]:
# visualize 2 image 
# your existing setup...
n_frame_show = n_pred_step
fields_all = field_dm
n_start = 0 
nframes = n_frame_show
total_duration = n_frame_show * 0.1

def update(i):
	i = i + n_start
	mat1.set_data(fields_all[i, 0, 0])
	mat2.set_data(fields_all[i, 1, 0])
	ax.set_title(f'pv_all_0 (idx = {i}) ')

fig, (ax1, ax2) = plt.subplots(nrows=1, ncols=2, figsize=(10, 5))
mat1 = ax1.imshow(fields_all[n_start,0, 0], vmax=1, vmin=-1)
mat2 = ax2.imshow(fields_all[n_start,1, 0], vmax=1, vmin=-1)
# fig.colorbar(mat0, ax=ax)
fig.tight_layout()



ani = animation.FuncAnimation(
	fig, update, frames=nframes, interval=int(1000.0 * total_duration / nframes)
)

plt.close(fig)

# Display the animation inline
HTML(ani.to_jshtml())  # or use ani.to_html5_video()


In [ ]:
def rmse(x):
	return np.sqrt(np.mean(x**2, axis=(1,2)))

field1 = fields_all[:, 0, 0,...]
field2 = fields_all[:, 1, 0,...]

In [ ]:
diff = field1 - field2
print(diff.shape)

diff_rmse = rmse(diff) / rmse(field2)

plt.plot(diff_rmse)
plt.show()

In [ ]:
field_diff = field1[1:, ...] - field1[:-1, ...]
print(field_diff.shape)

diff_rmse = rmse(field_diff) / rmse(field1[:-1])

plt.plot(diff_rmse)
plt.ylim(0, 1)
plt.show()

In [ ]:
# tau  = 1 
field_corr_gen = []
for tau in range(1, 50):
	field_corr_gen.append(np.mean(field1[tau:, ...] * field1[:-tau, ...]) )

plt.plot(field_corr_gen)
plt.show()

In [ ]:
n_sample_check = 500
feild_32 = []
with zipfile.ZipFile('/home/zoz/EDM-cond-turb/data/SQG/SQG_32_dynamic.zip', 'r') as zf:
	for i in range(n_sample_check):
		with zf.open(f'sample_{i:06d}.pt') as f:
			data = torch.load(io.BytesIO(f.read()))
			feild_32.append(data['image'])

feild_32 = torch.concat(feild_32, dim=0).numpy()
print(feild_32.shape)

# tau  = 1 
field_corr_true = []
# feild_32 = 
for tau in range(1, 50):
	v_cor = np.mean(feild_32[tau:, ...] * feild_32[:-tau, ...]) / 
	field_corr_true.append(np.mean(feild_32[tau:, ...] * feild_32[:-tau, ...]) )

plt.plot(field_corr_true)
plt.show()

In [ ]:
plt.plot(field_corr_true, label='true')
plt.plot(field_corr_gen, label='gen')
plt.legend()
plt.show()